## 1. 데이터 준비 및 전체 구조 파악

In [86]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np

In [87]:
# 데이터 불러오기
df = pd.read_csv('../dataset/raw/hotel_bookings.csv')
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [88]:
# 데이터 정보
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

## 2. 결측치 처리
### 2.1. 결측치 확인

In [89]:
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [90]:
# 'agent', 'company'는 숫자형 데이터 이지만 실제로는 코드값이므로 문자형으로 변환
cols = ['agent', 'company']

df[cols] = df[cols].astype(str)
df[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   agent    103050 non-null  str  
 1   company  6797 non-null    str  
dtypes: str(2)
memory usage: 1.8 MB


In [91]:
cols = ['children', 'country', 'agent', 'company']
df[cols].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
children,119386.0,NaN,NaN,NaN,0.10389,0.398561,0.0,0.0,0.0,0.0,10.0
country,118902,177,PRT,48590,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agent,103050,333,9.0,31961,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company,6797,352,40.0,927,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2.2. 기본전제
- 데이터 수집시 누락된 값 없이 수집되었음이 명시되었고
- 간혹 누락된 값이 있을 경우, 이 경우는 누락이 아닌 '해당없음'으로 간주하도록 안내되었다

### 2.3. 변수 종류에 따른 결측치 처리
- children: 미성년 수
    - 미성년이 없는 것으로 간주하여 0으로 채움
- country: 국적
    - 자국인 포루투갈로 예상되나 확실하지 않으므로 삭제
- agent: 예약을 진행한 여행사의 ID
    - 여행사를 거치지 않고 직접 예약을 진행한 것으로 간주
    - 0: 새 값 추가
- company: 예약을 진행했거나 예약 비용을 지불한 회사/기관의 ID
    - 예약을 대리해준 회사나 기관없이 직접 예약한 것으로 간주
    - 0: 새 값 추가

In [92]:
# 결측치 처리
cols_drop = ['country']
cols_fill = ['children', 'agent', 'company']

df.dropna(subset=cols_drop[0], axis=0, inplace=True)
df[cols_fill] = df[cols_fill].fillna(0)

df[cols_drop + cols_fill].isna().sum()

country     0
children    0
agent       0
company     0
dtype: int64

In [93]:
df[cols_fill].info()

<class 'pandas.DataFrame'>
Index: 118902 entries, 0 to 119389
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   children  118902 non-null  float64
 1   agent     118902 non-null  object 
 2   company   118902 non-null  object 
dtypes: float64(1), object(2)
memory usage: 3.6+ MB


## 3. 필요 함수 정의

In [52]:
# column 제거
# target: 제거할 column명, type - 리스트, 문자
def drop_target(target):
    df.drop(target, axis=1, inplace=True)

In [53]:
# value_counts 조회
# target: 조회할 column명, type - 리스트, 문자
def val_count(target):
    if type(target) == str:
        print(df[target].value_counts())
    else:
        for col in target:
            print(df[col].value_counts(), end='\n\n')

## 4. column 정리

### 4.1. 도착일 관련
- 대상: arrival_date_year, arrival_date_month, arrival_date_week_number, arrival_date_day_of_month
- 도착 연-월-일과 해당일의 주차가 각각의 column으로 분리되어 있음
- 월은 계절을 반영한 값으로 사용될 수 있으므로 유지
- 월을 제외한 나머지는 제거

In [54]:
cols = ['arrival_date_year', 'arrival_date_day_of_month', 'arrival_date_week_number']

drop_target(cols)
df.head()

,hotel,is_canceled,lead_time,arrival_date_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,July,0,0,2,0.0,0,BB,...,No Deposit,0,0,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,July,0,0,2,0.0,0,BB,...,No Deposit,0,0,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,July,0,1,1,0.0,0,BB,...,No Deposit,0,0,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,July,0,1,1,0.0,0,BB,...,No Deposit,304.0,0,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,July,0,2,2,0.0,0,BB,...,No Deposit,240.0,0,0,Transient,98.0,0,1,Check-Out,2015-07-03


### 4.2. 고객 관련
- 대상: adults, children, babies, customer_type

In [55]:
val_count('customer_type')

customer_type
Transient          89174
Transient-Party    25082
Contract            4076
Group                570
Name: count, dtype: int64


#### customer_type 유지
- 예약 유형(다음 네 가지 범주 중 하나로 분류):
    - 계약 - 예약에 할당량 또는 기타 유형의 계약이 있는 경우
    - 그룹 – 예약이 그룹과 연결된 경우
    - 개별 예약 – 단체 예약이나 계약에 포함되지 않고, 다른 개별 예약과 연계되지 않은 예약
    - 개별 예약(단체 예약 포함) - 개별 예약이지만, 하나 이상의 다른 개별 예약과 연결된 경우
- 투숙객 형태가 아닌 예약 형태로 분류된 값으로 투숙객 유형과는 성격이 다름

In [96]:
cols = ['adults', 'children', 'babies']
df[cols].describe(include='all').T

,count,mean,std,min,25%,50%,75%,max
adults,118902.0,1.858404,0.578576,0.0,2.0,2.0,2.0,55.0
children,118902.0,0.104203,0.399166,0.0,0.0,0.0,0.0,10.0
babies,118902.0,0.007948,0.097379,0.0,0.0,0.0,0.0,10.0


In [97]:
val_count(cols)

adults
2     89498
1     22735
3      6198
0       393
4        62
26        5
27        2
20        2
5         2
40        1
50        1
55        1
6         1
10        1
Name: count, dtype: int64

children
0.0     110323
1.0       4852
2.0       3650
3.0         76
10.0         1
Name: count, dtype: int64

babies
0     117988
1        898
2         14
10         1
9          1
Name: count, dtype: int64



In [103]:
# 구성 인원 수가 모두 0인 경우 조회
df.loc[df[cols].sum(axis=1) == 0].shape[0]

170

In [104]:
df.loc[df[cols].sum(axis=1) == 1].shape[0]

22289

In [106]:
# 어른 없이 1인인 경우 조회
df.loc[(df['adults'] == 0) & (df[cols].sum(axis=1) == 1)][cols]

,adults,children,babies
47110,0,1.0,0
48900,0,1.0,0
108456,0,1.0,0
113741,0,1.0,0


In [107]:
df.loc[(df['adults'] > 0) & (df['children'] + df['babies'] > 0)][cols]

,adults,children,babies
13,2,1.0,0
45,2,2.0,0
55,2,2.0,0
65,2,2.0,0
87,2,1.0,0
...,...,...,...
119270,2,1.0,0
119287,2,1.0,0
119293,2,2.0,0
119318,2,1.0,0


#### guest_type 추가
- 대상: adults, children, babies
    - 각각 성인, 아동, 유아 수를 의미
    - 세 열을 조합하여 guest_type 추가
- 값
    - Family: 성인 1인 이상 + 아동/유아가 1인 이상
    - Single: 1인으로만 구성
    - Group: 그 외 조합
        ex) 성인만 혹은 아동만 2인 이상 등
    - Undefined: 정보 없음

In [61]:
# 투숙객 유형을 분류하는 함수 정의
def classify_guest_type(row):
    adults = row['adults']
    children = row['children']
    babies = row['babies']
    total_kids = children + babies
    
    if adults + total_kids == 0:
        return 'Undefined'
    elif adults + total_kids == 1:
        return 'Single'
    elif adults > 0 and total_kids > 0:
        return 'Family'
    else:
        return 'Group'

In [62]:
df['guest_type'] = df[cols].apply(classify_guest_type, axis=1)
df[cols + ['guest_type']].head()

,adults,children,babies,guest_type
0,2,0.0,0,Group
1,2,0.0,0,Group
2,1,0.0,0,Single
3,1,0.0,0,Single
4,2,0.0,0,Group


In [63]:
df['guest_type'].value_counts()

guest_type
Group        87348
Single       22289
Family        9095
Undefined      170
Name: count, dtype: int64

In [64]:
drop_target(cols)
df.info()

<class 'pandas.DataFrame'>
Index: 118902 entries, 0 to 119389
Data columns (total 27 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           118902 non-null  str    
 1   is_canceled                     118902 non-null  int64  
 2   lead_time                       118902 non-null  int64  
 3   arrival_date_month              118902 non-null  str    
 4   stays_in_weekend_nights         118902 non-null  int64  
 5   stays_in_week_nights            118902 non-null  int64  
 6   meal                            118902 non-null  str    
 7   country                         118902 non-null  str    
 8   market_segment                  118902 non-null  str    
 9   distribution_channel            118902 non-null  str    
 10  is_repeated_guest               118902 non-null  int64  
 11  previous_cancellations          118902 non-null  int64  
 12  previous_bookings_not_canceled  

### 4.3. 객실 유형 관련 데이터
- 대상: 
    - reserved_room_type: 예약한 방
    - assigned_room_type: 실제 배정받은 방
    - 객실 유형을 코드로 나타낸 값으로 실제 객실 유형은 알 수 없으므로 diff_reserved_room_type 추가 후 삭제함
- diff_reserved_room_type 추가
    - 1: 예약한 방과 배정받은 방이 다른 경우
    - 0: 같은 경우

In [65]:
cols = ['reserved_room_type', 'assigned_room_type']

df['diff_reserved_room_type'] = (df[cols[0]] != df[cols[1]]).astype(int)
df[cols + ['diff_reserved_room_type']].head()

,reserved_room_type,assigned_room_type,diff_reserved_room_type
0,C,C,0
1,C,C,0
2,A,C,1
3,A,A,0
4,A,A,0


In [66]:
drop_target(cols)

### 4.4. 예약 유형 관련 데이터
- 대상
    - market_segment: 시장 세분화
    - distribution_channel: 유통 채널
    - agent: 예약을 진행한 여행사의 ID
    - company: 예약을 진행했거나 예약 비용을 지불한 회사/기관의 ID

In [67]:
cols = ['market_segment', 'distribution_channel', 'agent', 'company']

df[cols] = df[cols].astype(str)
df[cols].describe()

,market_segment,distribution_channel,agent,company
count,118902,118902,118902,118902
unique,8,5,333,350
top,Online TA,TA/TO,9.0,0
freq,56403,97730,31960,112279


In [68]:
val_count(cols)

market_segment
Online TA        56403
Offline TA/TO    24160
Groups           19806
Direct           12449
Corporate         5111
Complementary      734
Aviation           237
Undefined            2
Name: count, dtype: int64

distribution_channel
TA/TO        97730
Direct       14483
Corporate     6491
GDS            193
Undefined        5
Name: count, dtype: int64

agent
9.0      31960
0        16006
240.0    13871
1.0       7191
14.0      3639
         ...  
444.0        1
408.0        1
388.0        1
453.0        1
480.0        1
Name: count, Length: 333, dtype: int64

company
0        112279
40.0        927
223.0       784
67.0        267
45.0        250
          ...  
479.0         1
489.0         1
229.0         1
481.0         1
497.0         1
Name: count, Length: 350, dtype: int64



#### 처리 방향
- 유지: market_segment, distribution_channel
- 이진화: agent, company

In [69]:
df['is_agent'] = np.where(df['agent'] != '0', 1, 0)
df[['agent', 'is_agent']].head()

,agent,is_agent
0,0,0
1,0,0
2,0,0
3,304.0,1
4,240.0,1


In [70]:
df['is_company'] = np.where(df['company'] != '0', 1, 0)
df[['company', 'is_company']].head()

,company,is_company
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [71]:
drop_target(['agent', 'company'])

### 4.5. 예약 상태 관련 데이터
- 대상: reservation_status, reservation_status_date
    - 예약 최종 상태와 그 상태로 변경된 시점의 date
    - 예약 최종 상태 종류
        - Canceled
        - Check-Out: 예약 - 숙박 - 체크인 - 체크아웃
        - No-Show: 고객이 체크인하지 않았음
- 처리방안: target column인 is_canceled를 좀 더 세분화한 column으로 판단하여 삭제
    - Canceled | !Canceled --------------> is_canceled
    - !Canceled = Check-Out | No-Show

In [72]:
val_count('reservation_status')

reservation_status
Check-Out    74745
Canceled     42954
No-Show       1203
Name: count, dtype: int64


In [73]:
drop_target(['reservation_status', 'reservation_status_date'])

### 4.6. 기타
- hotel: 이진화

In [74]:
df['hotel'].unique()

<StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str

In [75]:
df['is_city_hotel'] = (df['hotel'] == 'City Hotel').astype(int)

In [76]:
drop_target(['hotel'])

- country
    - PRT: 포르투갈, 자국
    - EUR: 포르투갈외 유럽
    - OTH: !유럽

In [77]:
df['country'].unique()

<StringArray>
['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA', 'ROU', 'NOR', 'OMN', 'ARG',
 ...
 'ATA', 'GTM', 'ASM', 'MRT', 'NCL', 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 177, dtype: str

In [78]:
# 유럽 국가 코드
europe_codes = [
    'PRT', 'GBR', 'FRA', 'ESP', 'DEU', 'ITA', 'IRL', 'BEL', 'NLD', 'AUT', 
    'POL', 'SWE', 'CHE', 'NOR', 'DNK', 'FIN', 'CZE', 'GRC', 'ROU', 'HUN', 
    'HRV', 'SVK', 'SVN', 'BGR', 'LTU', 'LVA', 'EST', 'LUX', 'MLT', 'CYP'
]

def group_country(country):
    if country == 'PRT':
        return 'PRT'
    elif country in europe_codes:
        return 'EUR'  # 기타 유럽
    else:
        return 'OTH'  # 나머지

df['country_region'] = df['country'].apply(group_country)

In [79]:
df[['country', 'country_region']].head(15)

,country,country_region
0,PRT,PRT
1,PRT,PRT
2,GBR,EUR
3,GBR,EUR
4,GBR,EUR
5,GBR,EUR
6,PRT,PRT
7,PRT,PRT
8,PRT,PRT
9,PRT,PRT


In [80]:
val_count('country_region')

country_region
EUR    58312
PRT    48590
OTH    12000
Name: count, dtype: int64


In [81]:
drop_target(['country'])

## 마무리

In [82]:
df.info()

<class 'pandas.DataFrame'>
Index: 118902 entries, 0 to 119389
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   is_canceled                     118902 non-null  int64  
 1   lead_time                       118902 non-null  int64  
 2   arrival_date_month              118902 non-null  str    
 3   stays_in_weekend_nights         118902 non-null  int64  
 4   stays_in_week_nights            118902 non-null  int64  
 5   meal                            118902 non-null  str    
 6   market_segment                  118902 non-null  str    
 7   distribution_channel            118902 non-null  str    
 8   is_repeated_guest               118902 non-null  int64  
 9   previous_cancellations          118902 non-null  int64  
 10  previous_bookings_not_canceled  118902 non-null  int64  
 11  booking_changes                 118902 non-null  int64  
 12  deposit_type                    

In [83]:
# 파일 저장
df.to_csv('../dataset/preprocessed/hotel_bookings.csv', index=False)

In [84]:
dummy_cols = ['arrival_date_month', 'meal', 'market_segment','distribution_channel', 'deposit_type', 'customer_type', 'guest_type', 'country_region']

df = pd.get_dummies(df, columns=dummy_cols, drop_first=True, dtype=int)
df.head()

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr,...,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party,guest_type_Group,guest_type_Single,guest_type_Undefined,country_region_OTH,country_region_PRT
0,0,342,0,0,0,0,0,3,0,0.0,...,0,0,0,1,0,1,0,0,0,1
1,0,737,0,0,0,0,0,4,0,0.0,...,0,0,0,1,0,1,0,0,0,1
2,0,7,0,1,0,0,0,0,0,75.0,...,0,0,0,1,0,0,1,0,0,0
3,0,13,0,1,0,0,0,0,0,75.0,...,0,0,0,1,0,0,1,0,0,0
4,0,14,0,2,0,0,0,0,0,98.0,...,0,0,0,1,0,1,0,0,0,0


In [85]:
# 파일 저장
df.to_csv('../dataset/preprocessed/hotel_bookings_dummy.csv', index=False)